# RSNA Knee Abnormality Detection — Weak-Label Evaluation

Measures `extract_weak_labels` (assertion-aware) against `extract_weak_labels_naive` (the original, keyword-only extractor) on the 58 human-labeled studies, and produces a per-label allowlist of which labels are trustworthy enough to weak-label the remaining 4,349 report-only studies with. This notebook is committed output-free, always — its cells process real report text directly. Only aggregate counts and rates are ever shown; no report excerpts, no per-study prediction tables, no study-identifier lists.

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

    SRC_DATASET_DIR = Path("/kaggle/input/datasets/tuannm3812/rsna-knee-mri-src")
    _src_candidates = (SRC_DATASET_DIR / "src", SRC_DATASET_DIR)
    _src_root = next((c for c in _src_candidates if (c / "knee_mri").is_dir()), None)
    if _src_root is None:
        raise RuntimeError(
            "knee_mri package not found under the attached rsna-knee-mri-src dataset."
        )
    sys.path.insert(0, str(_src_root))
else:
    raise RuntimeError(
        "This notebook only runs on Kaggle -- the competition dataset is "
        "never downloaded locally."
    )

## 1. Frozen Evaluation Contract

In [ ]:
from knee_mri.dataset import split_labeled_studies
from knee_mri.weak_label_evaluation import MIN_PRECISION_LOWER_BOUND, MIN_SUPPORT

train_df = pd.read_csv(DATA_DIR / "train.csv")
labeled_df, unlabeled_df = split_labeled_studies(train_df)

study_counts = pd.Series(
    {
        "Labeled studies": len(labeled_df),
        "Unlabeled (report-only) studies": len(unlabeled_df),
    },
    name="Count",
).to_frame()
decision_rule = pd.Series(
    {
        "Minimum predicted-positive support": MIN_SUPPORT,
        "Minimum Wilson lower-bound precision": MIN_PRECISION_LOWER_BOUND,
    },
    name="Frozen Threshold",
).to_frame()

display(study_counts)
display(decision_rule)

**Interpretation.** This evaluation scores two frozen extractors against the 58 human-labeled studies; the remaining 4,349 studies are report-only and held out from this pass. A label is allowlisted only if its Wilson lower-bound precision reaches 0.55 with at least 5 predicted positives — both thresholds were frozen before any result was viewed, per the design spec's decision rule.

## 2. Naive Keyword Baseline

In [ ]:
from knee_mri.labels import extract_weak_labels, extract_weak_labels_naive
from knee_mri.weak_label_evaluation import weak_label_metrics

baseline_metrics = weak_label_metrics(labeled_df, extract_weak_labels_naive)
display(baseline_metrics)

**Interpretation.** The naive extractor never abstains, so coverage is 1.0 for every label. Precision is weak across most labels (e.g. ACL 0.414, MCL 0.200) and no label's Wilson lower bound clears the frozen 0.55 threshold.

## 3. Assertion-Aware Extractor

In [ ]:
fixed_metrics = weak_label_metrics(labeled_df, extract_weak_labels)
display(fixed_metrics)

**Interpretation.** Assertion awareness improves point-estimate precision for most labels by abstaining rather than forcing a guess — e.g. Medial Meniscus 0.545→0.750, Lateral Meniscus 0.524→0.769, Fracture 0.500→1.000 — but no label's Wilson lower bound/support pair clears the frozen gate at this sample size. Closest misses: Medial Meniscus (lower bound 0.505, support 16 — clears the support floor but not the precision bound) and Fracture (lower bound 0.510, but support only 4, short of the minimum 5 regardless of precision).

## 4. Coverage and Error Taxonomy

In [ ]:
from collections import Counter

from knee_mri.labels import LABEL_COLUMNS, _resolution_signature, _resolve_weak_labels
from knee_mri.weak_label_evaluation import orthographic_bucket

taxonomy_counts = Counter()
for _, row in labeled_df.iterrows():
    bucket = orthographic_bucket(row["Report"])
    resolutions = _resolve_weak_labels(row["Report"])
    for label in LABEL_COLUMNS:
        resolution = resolutions[label]
        truth = row[label]
        prediction = resolution.value
        is_error = (prediction == 1 and truth == 0) or (prediction != 1 and truth == 1)
        if not is_error:
            continue
        prediction_error = "false_positive" if prediction == 1 else "false_negative"
        signature = _resolution_signature(resolution.mentions)
        taxonomy_counts[(label, bucket, signature, prediction_error)] += 1

# Counts only -- never report text, matched text, or per-study identifiers.
taxonomy_table = pd.DataFrame(
    [
        {
            "Label": label,
            "Orthographic Bucket": bucket,
            "Resolution Signature": signature,
            "Prediction Error": error,
            "Count": count,
        }
        for (label, bucket, signature, error), count in sorted(taxonomy_counts.items())
    ]
)
display(taxonomy_table)

**Interpretation.** `no_mention` (no keyword matched at all, so the extractor correctly abstained rather than guessing) dominates false negatives across almost every label and non-ASCII bucket — this is why coverage drops well below 1.0 for most labels in Section 3. `unqualified_only` (a keyword matched with no qualifying cue nearby, correctly resolved to 1) accounts for most false positives and is concentrated in `ascii_only`. These are directly observed mechanical facts, not a proven cause — a causal read (e.g. that the keyword vocabulary's English-only scope under-covers non-English reports) is a plausible hypothesis, not something this table establishes on its own.

## 5. Labeled-to-Unlabeled Orthographic Comparison

In [ ]:
labeled_buckets = (
    labeled_df["Report"].dropna().apply(orthographic_bucket).value_counts(normalize=True)
)
unlabeled_buckets = (
    unlabeled_df["Report"].dropna().apply(orthographic_bucket).value_counts(normalize=True)
)
comparison = pd.DataFrame(
    {"labeled": labeled_buckets, "unlabeled": unlabeled_buckets}
).fillna(0.0)
comparison["gap_pp"] = (comparison["labeled"] - comparison["unlabeled"]).abs() * 100

display(comparison)

**Interpretation.** The `ascii_only` bucket has a real 7.7 percentage-point gap between the 58 labeled studies and the 4,349 unlabeled studies (labeled studies skew more English than the unlabeled population); the other five buckets are within 1.7–2.8 points. Not close enough across the board to call the labeled sample's character-set mix representative — and even a close mix would only bound how similar the text looks, not how well extraction accuracy transfers to non-English reports.

## 6. Decision and Modeling Implication

In [ ]:
allowlist = fixed_metrics[fixed_metrics["passes_gate"]].index.tolist()
allowlist_summary = pd.Series(
    {"Labels evaluated": len(LABEL_COLUMNS), "Labels on allowlist": len(allowlist)},
    name="Count",
).to_frame()

display(allowlist_summary)

**Interpretation and decision: No-go.** The allowlist is empty — 0/12 labels clear the frozen gate at this sample size. Phase 3A therefore trains only on the 58 human-labeled studies; no weak-labeled studies are added to the training set from this pass. This is a real fork, not a formality: a future pass could reopen weak supervision with a multilingual or probabilistic approach, but that is a separate, not-yet-scoped decision.